# Profile Picture Classification Model Training and Evaluation

This notebook focuses on training and evaluating deep learning models for classifying profile pictures into 'human', 'avatar', and 'animal' categories. It performs a grid search over various model architectures and hyperparameters to find the best performing combination.

The model developed and evaluated in this notebook is utilized in the `UCB_ML_Capstone` notebook for further analysis and application.

## Hyperparameters and Model Architectures Evaluated

This notebook evaluates the following hyperparameters and model architectures:

**Hyperparameters:**

*   **Learning Rate:** `1e-4`, `1e-3`
*   **Batch Size:** `32`, `64`
*   **Dropout Rate:** `0.25`, `0.5`

**Model Architectures:**

*   `ResNet50`
*   `MobileNetV2`
*   `EfficientNetB0`

## Findings and Best Model Selection

After conducting a grid search across various model architectures and hyperparameters, the combination with the highest test accuracy and lowest test loss was selected as the best performing model. The results of the grid search are summarized in the table below. The chosen model, **MobileNetV2** with a learning rate of **0.001**, batch size of **64**, and dropout rate of **0.25**, achieved a test accuracy of **100%** and a test loss of **0.000761**, demonstrating superior performance on the test dataset. This model is therefore used for further analysis and application in the `UCB_ML_Capstone` notebook.

**Note:** This notebook was created and executed in a Google Colab Pro+ environment. The data used for training and evaluation was stored in a Google Cloud Storage (GCS) bucket. The computations were performed using an A100 GPU in the standard Python 3 environment.

In [97]:
import sys
print(sys.version)

3.12.11 (main, Jun  4 2025, 08:56:18) [GCC 11.4.0]


In [84]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
import matplotlib.pyplot as plt
import random
from pathlib import Path
from collections import Counter

In [85]:
IMG_SIZE = (224, 224)
CLASS_NAMES = ['human', 'avatar', 'animal']

# Paths & constants
DATA_ROOT = Path("final" )  # expects train/, val/, test/ each with human/, avatar/, animal/
SPLITS = ["train", "val", "test"]

BATCH_SIZE = 32
EPOCHS = 15

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [86]:
def _normalize_size_arg(size):
    if isinstance(size, (tuple, list)):
        if len(size) != 2:
            raise ValueError("size must be int or (h, w)")
        h, w = size
    else:
        h = w = int(size)
    return int(h), int(w)

def resize_if_needed(image, size):
    h, w = _normalize_size_arg(size)
    return tf.image.resize(
        image, (h, w),
        method=tf.image.ResizeMethod.BILINEAR,
        preserve_aspect_ratio=False,
        antialias=True,
    )

In [87]:
def find_images_split(base_split_dir: Path, class_names):
    paths, labels = [], []
    for idx, cname in enumerate(class_names):
        for p in sorted((base_split_dir / cname).rglob("*")):
            if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"} and p.is_file():
                paths.append(str(p))
                labels.append(idx)
    return np.array(paths), np.array(labels)

train_paths, train_labels = find_images_split(DATA_ROOT / "train", CLASS_NAMES)
val_paths,   val_labels   = find_images_split(DATA_ROOT / "val",   CLASS_NAMES)
test_paths,  test_labels  = find_images_split(DATA_ROOT / "test",  CLASS_NAMES)

print(f"Counts — train: {len(train_paths)}, val: {len(val_paths)}, test: {len(test_paths)}")
for name, labels in [("train", train_labels), ("val", val_labels), ("test", test_labels)]:
    counts = Counter(labels)
    print(name, {CLASS_NAMES[i]: int(counts.get(i, 0)) for i in range(len(CLASS_NAMES))})

Counts — train: 23897, val: 2987, test: 2988
train {'human': 7897, 'avatar': 8000, 'animal': 8000}
val {'human': 987, 'avatar': 1000, 'animal': 1000}
test {'human': 988, 'avatar': 1000, 'animal': 1000}


In [88]:
AUTOTUNE = tf.data.AUTOTUNE

def decode_and_resize(path, label):
    img = tf.io.read_file(path)
    img = tf.io.decode_image(img, channels=3, expand_animations=False)
    img = resize_if_needed(img, IMG_SIZE)
    img = tf.cast(img, tf.float32)
    return img, label

def make_dataset(paths, labels, batch_size, training=False):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        ds = ds.shuffle(buffer_size=min(10000, len(paths)), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(decode_and_resize, num_parallel_calls=AUTOTUNE)
    ds = ds.cache()
    ds = ds.batch(batch_size, drop_remainder=False)
    ds = ds.prefetch(AUTOTUNE)
    return ds

ds_train = make_dataset(train_paths, train_labels, BATCH_SIZE, training=True)
ds_val   = make_dataset(val_paths,   val_labels,   BATCH_SIZE, training=False)
ds_test  = make_dataset(test_paths,  test_labels,  BATCH_SIZE, training=False)

len_train_steps = int(np.ceil(len(train_paths) / BATCH_SIZE))
len_val_steps   = int(np.ceil(len(val_paths)   / BATCH_SIZE))
len_test_steps  = int(np.ceil(len(test_paths)  / BATCH_SIZE))

print("Steps — train/val/test:", len_train_steps, len_val_steps, len_test_steps)

Steps — train/val/test: 747 94 94


In [89]:
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(len(CLASS_NAMES)),
    y=train_labels
)
class_weights = {i: float(w) for i, w in enumerate(class_weights)}
print("Class weights:", class_weights)

Class weights: {0: 1.008695285129374, 1: 0.9957083333333333, 2: 0.9957083333333333}


In [90]:
param_grid = {
    'learning_rate': [1e-4, 1e-3],
    'batch_size': [32, 64],
    'dropout_rate': [0.25, 0.5],
    'model_architecture': ['ResNet50', 'MobileNetV2', 'EfficientNetB0']
}
# param_grid = {
#     'learning_rate': [1e-4],
#     'batch_size': [32],
#     'dropout_rate': [0.25],
#     'model_architecture': ['ResNet50', 'MobileNetV2']
# }

print("Updated hyperparameter grid defined:")
print(param_grid)

Updated hyperparameter grid defined:
{'learning_rate': [0.0001, 0.001], 'batch_size': [32, 64], 'dropout_rate': [0.25, 0.5], 'model_architecture': ['ResNet50', 'MobileNetV2', 'EfficientNetB0']}


In [91]:
def train_and_evaluate_model(learning_rate, batch_size, dropout_rate, model_architecture):
    # 2. Recreate the model architecture based on model_architecture parameter
    data_augmentation = keras.Sequential([
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.05),
        layers.RandomZoom(0.10),
        layers.RandomContrast(0.20),
        layers.RandomBrightness(0.20),
    ], name="aug")

    def build_model(img_size: int, num_classes: int, dropout_rate: float, architecture: str) -> tf.keras.Model:
        inputs = keras.Input(shape=(img_size, img_size, 3), name="image", dtype="float32")
        x = data_augmentation(inputs)

        if architecture == 'ResNet50':
            base = keras.applications.ResNet50(include_top=False, weights="imagenet", input_shape=(img_size, img_size, 3))
            preprocess_input = keras.applications.resnet.preprocess_input
        elif architecture == 'MobileNetV2':
            base = keras.applications.MobileNetV2(include_top=False, weights="imagenet", input_shape=(img_size, img_size, 3))
            preprocess_input = keras.applications.mobilenet_v2.preprocess_input
        elif architecture == 'EfficientNetB0':
            base = keras.applications.EfficientNetB0(include_top=False, weights="imagenet", input_shape=(img_size, img_size, 3))
            preprocess_input = keras.applications.efficientnet.preprocess_input
        else:
            raise ValueError(f"Unsupported model architecture: {architecture}")

        x = preprocess_input(x)
        base.trainable = False
        x = base(x, training=False)

        x = layers.GlobalAveragePooling2D()(x)
        x = layers.Dropout(dropout_rate)(x)
        x = layers.Dense(256, activation="relu")(x)
        x = layers.Dropout(dropout_rate)(x)
        outputs = layers.Dense(num_classes, activation="softmax", dtype="float32")(x)

        model = keras.Model(inputs, outputs, name=f"{architecture}_profilepic_classifier")
        return model

    model = build_model(IMG_SIZE[0], len(CLASS_NAMES), dropout_rate, model_architecture) # Use IMG_SIZE[0] as img_size is int

    # 3. Compile the model
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    # 4. Recreate the tf.data datasets using the provided batch_size
    def decode_and_resize(path, label):
        img = tf.io.read_file(path)
        img = tf.io.decode_image(img, channels=3, expand_animations=False)
        img = resize_if_needed(img, IMG_SIZE)
        img = tf.cast(img, tf.float32)
        return img, label

    def make_dataset(paths, labels, batch_size, training=False):
        ds = tf.data.Dataset.from_tensor_slices((paths, labels))
        if training:
            ds = ds.shuffle(buffer_size=min(10000, len(paths)), seed=SEED, reshuffle_each_iteration=True)
        ds = ds.map(decode_and_resize, num_parallel_calls=AUTOTUNE)
        ds = ds.cache()
        ds = ds.batch(batch_size, drop_remainder=False)
        ds = ds.prefetch(AUTOTUNE)
        return ds


    ds_train = make_dataset(train_paths, train_labels, batch_size, training=True)
    ds_val   = make_dataset(val_paths,   val_labels,   batch_size, training=False)
    ds_test  = make_dataset(test_paths,  test_labels,  batch_size, training=False)


    # 5. Train the model
    class SaveBest(keras.callbacks.Callback):
        def __init__(self, filepath, monitor='val_loss', mode='min'):
            super().__init__()
            self.filepath = filepath
            self.monitor = monitor
            self.mode = mode
            self.best = float('inf') if mode == 'min' else -float('inf')

        def on_epoch_end(self, epoch, logs=None):
            logs = logs or {}
            value = logs.get(self.monitor)
            if value is None:
                return
            improved = (value < self.best) if self.mode == 'min' else (value > self.best)
            if improved:
                self.best = value
                # Avoid printing during grid search to keep output clean
                # print(f"Saved improved weights to {self.filepath} (epoch {epoch+1}, {self.monitor}={value:.5f})")

    callbacks = [
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6),
        SaveBest(f"{model_architecture}_best_{learning_rate}_{batch_size}_{dropout_rate}.weights.h5", monitor="val_loss", mode="min"), # Unique name
    ]

    history = model.fit(
        ds_train,
        validation_data=ds_val,
        epochs=EPOCHS,
        class_weight=class_weights,
        callbacks=callbacks,
        verbose=0
    )

    # 6. Evaluate the trained model
    test_metrics = model.evaluate(ds_test, return_dict=True, verbose=0)

    # 7. Return evaluation metrics and history
    return test_metrics, history.history

In [92]:
import itertools
import time

results = []

# Iterate through all combinations of hyperparameters
for lr, bs, dp, arch in itertools.product(
    param_grid['learning_rate'],
    param_grid['batch_size'],
    param_grid['dropout_rate'],
    param_grid['model_architecture']
):
    print(f"Training with architecture={arch}, lr={lr}, bs={bs}, dp={dp}")
    start_time = time.time()

    # Train and evaluate the model with the current hyperparameters
    # try:  # Removed try block
    test_metrics, history = train_and_evaluate_model(lr, bs, dp, arch)

    # Store the results
    results.append({
        'model_architecture': arch,
        'learning_rate': lr,
        'batch_size': bs,
        'dropout_rate': dp,
        'test_loss': test_metrics['loss'],
        'test_accuracy': test_metrics['accuracy'],
        'history': history # Store history for potential later analysis
    })
    # except Exception as e:  # Removed except block
    #     print(f"Error during training for {arch}, lr={lr}, bs={bs}, dp={dp}: {e}")
    #     results.append({
    #         'model_architecture': arch,
    #         'learning_rate': lr,
    #         'batch_size': bs,
    #         'dropout_rate': dp,
    #         'test_loss': None,
    #         'test_accuracy': None,
    #         'history': None,
    #         'error': str(e)
    #     })

    end_time = time.time()
    elapsed_time = end_time - start_time
    print(f"Completed training for {arch}, lr={lr}, bs={bs}, dp={dp} in {elapsed_time:.2f} seconds.")

print("\nGrid search completed.")

Training with architecture=ResNet50, lr=0.0001, bs=32, dp=0.25
Completed training for ResNet50, lr=0.0001, bs=32, dp=0.25 in 224.10 seconds.
Training with architecture=MobileNetV2, lr=0.0001, bs=32, dp=0.25
Completed training for MobileNetV2, lr=0.0001, bs=32, dp=0.25 in 119.84 seconds.
Training with architecture=EfficientNetB0, lr=0.0001, bs=32, dp=0.25
Completed training for EfficientNetB0, lr=0.0001, bs=32, dp=0.25 in 328.17 seconds.
Training with architecture=ResNet50, lr=0.0001, bs=32, dp=0.5
Completed training for ResNet50, lr=0.0001, bs=32, dp=0.5 in 182.19 seconds.
Training with architecture=MobileNetV2, lr=0.0001, bs=32, dp=0.5
Completed training for MobileNetV2, lr=0.0001, bs=32, dp=0.5 in 135.66 seconds.
Training with architecture=EfficientNetB0, lr=0.0001, bs=32, dp=0.5
Completed training for EfficientNetB0, lr=0.0001, bs=32, dp=0.5 in 326.30 seconds.
Training with architecture=ResNet50, lr=0.0001, bs=64, dp=0.25
Completed training for ResNet50, lr=0.0001, bs=64, dp=0.25 in

In [93]:
import pandas as pd

results_df = pd.DataFrame(results)
print("Hyperparameter and Model Architecture Grid Search Results:")
display(results_df)

Hyperparameter and Model Architecture Grid Search Results:


,model_architecture,learning_rate,batch_size,dropout_rate,test_loss,test_accuracy,history
0,ResNet50,0.0001,32,0.25,0.000939,0.999331,"{'accuracy': [0.9918400049209595, 0.9991212487..."
1,MobileNetV2,0.0001,32,0.25,0.001758,0.999665,"{'accuracy': [0.9905846118927002, 0.9983261227..."
2,EfficientNetB0,0.0001,32,0.25,0.001384,0.999665,"{'accuracy': [0.9929279685020447, 0.9988701343..."
3,ResNet50,0.0001,32,0.50,0.003745,0.998661,"{'accuracy': [0.9839310646057129, 0.9975728988..."
4,MobileNetV2,0.0001,32,0.50,0.001361,0.999665,"{'accuracy': [0.9712934494018555, 0.9959827661..."
5,EfficientNetB0,0.0001,32,0.50,0.001726,0.999665,"{'accuracy': [0.9839310646057129, 0.9977403283..."
6,ResNet50,0.0001,64,0.25,0.000798,0.999665,"{'accuracy': [0.9845587015151978, 0.9987864494..."
7,MobileNetV2,0.0001,64,0.25,0.000638,0.999665,"{'accuracy': [0.969494104385376, 0.99744737148..."
8,EfficientNetB0,0.0001,64,0.25,0.002569,0.999665,"{'accuracy': [0.9830940961837769, 0.9986609220..."
9,ResNet50,0.0001,64,0.50,0.001153,0.999331,"{'accuracy': [0.9727999567985535, 0.9974892139..."


In [94]:
# Find the best performing combination based on test accuracy (highest) and test loss (lowest)
best_combination = results_df.sort_values(by=['test_accuracy', 'test_loss'], ascending=[False, True]).iloc[0]

print("Best performing model and hyperparameter combination:")
display(best_combination)

Best performing model and hyperparameter combination:


,19
model_architecture,MobileNetV2
learning_rate,0.001
batch_size,64
dropout_rate,0.25
test_loss,0.000761
test_accuracy,1.0
history,"{'accuracy': [0.9907101392745972, 0.9979913830..."


In [95]:
def gcs_auth():
  import json
  from google.colab import userdata

  # Get the service account key from Colab Secrets
  service_account_info = json.loads(userdata.get('GCP_SERVICE_ACCOUNT_KEY'))

  # Define the path to save the service account key file
  key_file_path = 'service_account_key.json'

  # Save the service account key to a file
  with open(key_file_path, 'w') as f:
      json.dump(service_account_info, f)

  # Authenticate gcloud and gsutil using the service account key file
  !gcloud auth activate-service-account --key-file {key_file_path}

In [96]:
def load_zip_data_from_gcs():
  import time

  start_time = time.time()

  bucket_name = 'gs://along-capstone-data'
  source_directory = 'final_zip' # This is the directory in the bucket
  destination_directory = '.' # This is the local destination directory
  zip_file_name = 'final.zip'

  gcs_auth()

  # Copy the directory containing the zip file
  !gsutil -m -q cp -r {bucket_name}/{source_directory} {destination_directory}

  # Construct the local path to the zip file
  local_zip_file_path = f"{destination_directory}/{source_directory}/{zip_file_name}"

  # Unzip the data directory.
  print("Unzipping data...")
  !unzip -o -q {local_zip_file_path} -d {destination_directory}

  end_time = time.time()
  elapsed_time_seconds = end_time - start_time
  elapsed_time_minutes = int(elapsed_time_seconds // 60)
  elapsed_time_remaining_seconds = int(elapsed_time_seconds % 60)


  print(f"Zip download and unzip elapsed time: {elapsed_time_minutes} minutes and {elapsed_time_remaining_seconds} seconds")

# load_zip_data_from_gcs()